# Test Scala Data Provenance

### Prerequisites

In [31]:
import $ivy.`org.apache.spark::spark-sql:4.1.1`

import $ivy.$

In [32]:
val version = scala.io.Source.fromFile("../VERSION")  // Get version from file
  .getLines().next().trim

interp.load.ivy("org.dataprov.dp" %% "dp-spark" % version)  // use porogrammatic API 

// // For publishLocal (~/.ivy2/local)
// import $ivy.`org.dataprov.dp::dp-spark:0.0.1` // N.B: version must be specified explicitly with this method

// // For publishM2 (~/.m2)
// import $repo.`file:///home/ronan/.m2/repository`
// import $ivy.`org.dataprov.dp::dp-spark:0.0.1` // N.B: version must be specified explicitly with this method

version: String = "0.0.1"

### Import libraries

In [33]:
import java.sql.Date

import org.apache.spark.sql.{SparkSession, DataFrame, Dataset}
import org.apache.spark.sql.functions._
import org.apache.spark.sql.execution.SparkPlan

import org.apache.spark.sql.catalyst.plans.logical._
import org.apache.spark.sql.catalyst.expressions._
import org.apache.spark.sql.catalyst.expressions.aggregate._

import org.dataprov.dp.sparkdataprovenance.DataFrameProvenanceTransformations._

import java.sql.Date
import org.apache.spark.sql.{SparkSession, DataFrame, Dataset}
import org.apache.spark.sql.functions._
import org.apache.spark.sql.execution.SparkPlan
import org.apache.spark.sql.catalyst.plans.logical._
import org.apache.spark.sql.catalyst.expressions._
import org.apache.spark.sql.catalyst.expressions.aggregate._
import org.dataprov.dp.sparkdataprovenance.DataFrameProvenanceTransformations._

### Initialize spark session

In [34]:
val spark = SparkSession.builder()
  .master("local[*]")
  .appName("notebook")
  .getOrCreate()

// Set log level to ERROR to reduce verbosity
spark.sparkContext.setLogLevel("ERROR")

spark: SparkSession = org.apache.spark.sql.classic.SparkSession@2c7c439

### Create spark dataframe

We create a dataframe with duplicates to test the provenance annotation.
 
The provenance annotation will count the number of duplicates for each tuple.

In [35]:
val df: DataFrame = spark.createDataFrame(
    Seq(
        ("a", "b", "c"),
        ("a", "b", "c"),
        ("d", "b", "e"),
        ("d", "b", "e"),
        ("d", "b", "e"),
        ("d", "b", "e"),
        ("d", "b", "e"),
        ("f", "g", "e")
    )
).toDF("A", "B", "C")

df.show()

+---+---+---+
|  A|  B|  C|
+---+---+---+
|  a|  b|  c|
|  a|  b|  c|
|  d|  b|  e|
|  d|  b|  e|
|  d|  b|  e|
|  d|  b|  e|
|  d|  b|  e|
|  f|  g|  e|
+---+---+---+



df: DataFrame = [A: string, B: string ... 1 more field]

We create another table with annotations.

In this case, the annotations are natural numbers which represent the multiplicity of the tuple in the multiset. 

We can compute the provenance annotation by grouping by the columns and counting the number of duplicates for each tuple.


In [36]:
val dfWithProvenance: DataFrame = df
    .groupBy("A", "B", "C")
    .count()
    .withColumnRenamed("count", "provenance")
    .orderBy("A", "B", "C")
    
dfWithProvenance.show()


+---+---+---+----------+
|  A|  B|  C|provenance|
+---+---+---+----------+
|  a|  b|  c|         2|
|  d|  b|  e|         5|
|  f|  g|  e|         1|
+---+---+---+----------+



dfWithProvenance: DataFrame = [A: string, B: string ... 2 more fields]

In [37]:
// 1. Parsed Logical Plan (Unresolved)
val parsedPlan: LogicalPlan = df.queryExecution.logical

// 2. Analyzed Logical Plan (Resolved)
// This is usually what you want when inspecting the schema and resolved columns
val analyzedPlan: LogicalPlan = df.queryExecution.analyzed

// 3. Optimized Logical Plan
// This is the plan after Spark's Catalyst optimizer has applied its rules
val optimizedPlan: LogicalPlan = df.queryExecution.optimizedPlan

// 4. Physical Plan
// This is the plan produced by the physical planning strategies, 
// but BEFORE rule-based physical optimizations (like whole-stage codegen) are applied.
val physicalPlan: SparkPlan = df.queryExecution.sparkPlan

// 5. Executed Physical Plan (Final)
// This is the absolute final plan that Spark actually runs. It includes 
// execution-specific preparations like adding Exchange nodes (shuffles) 
// and WholeStageCodegen blocks.
val executedPlan: SparkPlan = df.queryExecution.executedPlan

parsedPlan: LogicalPlan = UnresolvedSubqueryColumnAliases(
  outputColumnNames = ArraySeq("A", "B", "C"),
  child = LocalRelation(
    output = List(
      AttributeReference(
        name = "_1",
        dataType = StringType,
        nullable = true,
        metadata = {}
      ),
      AttributeReference(
        name = "_2",
        dataType = StringType,
        nullable = true,
        metadata = {}
      ),
      AttributeReference(
        name = "_3",
        dataType = StringType,
        nullable = true,
        metadata = {}
      )
    ),
    data = List(
      [a,b,c],
      [a,b,c],
      [d,b,e],
      [d,b,e],
      [d,b,e],
      [d,b,e],
      [d,b,e],
      [f,g,e]
    ),
    isStreaming = false,
    stream = None
  )
)
analyzedPlan: LogicalPlan = Project(
  projectList = List(
    Alias(
      child = AttributeReference(
        name = "_1",
        dataType = StringType,
        nullable = true,
        metadata = {}
      ),
      name = "A"
    ),
    Alias(
   

## Bag semiring

### Example $ \pi_{AB}R \bowtie \pi_{BC}R$ 

In [38]:
// Expected results:
// +---+---+---+----------+
// |  A|  B|  C|provenance|
// +---+---+---+----------+
// |  a|  b|  c|         4| 2*2
// |  a|  b|  e|        10| 2*5
// |  d|  b|  c|        10| 5*2
// |  d|  b|  e|        25| 5*5
// |  f|  g|  e|         1| 1*1
// +---+---+---+----------+

val df2: DataFrame = df
    .select("A", "B")
    .join(df.select("B", "C"), "B")
    .select("A", "B", "C")
    .orderBy("A", "B", "C")

df2.show()

val leftSide = dfWithProvenance
  .groupBy("A", "B")
  .agg(sum("provenance").alias("count_A")) 

val rightSide = dfWithProvenance
  .groupBy("B", "C")
  .agg(sum("provenance").alias("count_C"))

val df2WithProvenance: DataFrame = leftSide
  .join(rightSide, Seq("B"))
  .withColumn("provenance", col("count_A") * col("count_C")) 
  .drop("count_A", "count_C")
  .select("A", "B", "C", "provenance")
  .orderBy("A", "B", "C")

df2WithProvenance.show()

+---+---+---+
|  A|  B|  C|
+---+---+---+
|  a|  b|  c|
|  a|  b|  c|
|  a|  b|  c|
|  a|  b|  c|
|  a|  b|  e|
|  a|  b|  e|
|  a|  b|  e|
|  a|  b|  e|
|  a|  b|  e|
|  a|  b|  e|
|  a|  b|  e|
|  a|  b|  e|
|  a|  b|  e|
|  a|  b|  e|
|  d|  b|  c|
|  d|  b|  c|
|  d|  b|  c|
|  d|  b|  c|
|  d|  b|  c|
|  d|  b|  c|
+---+---+---+
only showing top 20 rows
+---+---+---+----------+
|  A|  B|  C|provenance|
+---+---+---+----------+
|  a|  b|  c|         4|
|  a|  b|  e|        10|
|  d|  b|  c|        10|
|  d|  b|  e|        25|
|  f|  g|  e|         1|
+---+---+---+----------+



df2: DataFrame = [A: string, B: string ... 1 more field]
leftSide: DataFrame = [A: string, B: string ... 1 more field]
rightSide: DataFrame = [B: string, C: string ... 1 more field]
df2WithProvenance: DataFrame = [A: string, B: string ... 2 more fields]

Without provenance
```
== Optimized Logical Plan ==
Sort [A#3 ASC NULLS FIRST, B#4 ASC NULLS FIRST, C#1195 ASC NULLS FIRST], true
+- Project [A#3, B#4, C#1195]
   +- Join Inner, (B#4 = B#1194)
      :- LocalRelation [A#3, B#4]
      +- LocalRelation [B#1194, C#1195]
```

With provenance
```     
== Optimized Logical Plan ==
Sort [A#3 ASC NULLS FIRST, B#4 ASC NULLS FIRST, C#1223 ASC NULLS FIRST], true
+- Project [A#3, B#4, C#1223, (count_A#1206L * count_C#1212L) AS provenance#1226L]
   +- Join Inner, (B#4 = B#1222)
      :- Aggregate [A#3, B#4], [A#3, B#4, sum(provenance#675L) AS count_A#1206L]
      :  +- Aggregate [A#3, B#4, C#5], [A#3, B#4, count(1) AS provenance#675L]
      :     +- LocalRelation [A#3, B#4, C#5]
      +- Aggregate [B#1222, C#1223], [B#1222, C#1223, sum(provenance#1225L) AS count_C#1212L]
         +- Aggregate [A#1221, B#1222, C#1223], [B#1222, C#1223, count(1) AS provenance#1225L]
            +- LocalRelation [A#1221, B#1222, C#1223]
```  

### Example $ \pi_{AC}R \bowtie \pi_{BC}R$ 

In [39]:
// Expected results:
// +---+---+---+----------+
// |  A|  B|  C|provenance|
// +---+---+---+----------+
// |  a|  b|  c|         4|
// |  d|  b|  e|        25|
// |  d|  g|  e|         5|
// |  f|  b|  e|         5|
// |  f|  g|  e|         1|
// +---+---+---+----------+

val df3 : DataFrame = df
  .select("A", "C")
  .join(df.select("B", "C"), "C")
  .select("A", "B", "C")
  .orderBy("A", "B", "C")


val leftSide = dfWithProvenance
  .groupBy("A", "C")
  .agg(sum("provenance").alias("count_A")) 

val rightSide = dfWithProvenance
  .groupBy("B", "C")
  .agg(sum("provenance").alias("count_B"))

val df3WithProvenance: DataFrame = leftSide
  .join(rightSide, Seq("C"))
  .withColumn("provenance", col("count_A") * col("count_B"))
  .drop("count_A", "count_B")
  .select("A", "B", "C", "provenance")
  .orderBy("A", "B", "C")

df3WithProvenance.show()

+---+---+---+----------+
|  A|  B|  C|provenance|
+---+---+---+----------+
|  a|  b|  c|         4|
|  d|  b|  e|        25|
|  d|  g|  e|         5|
|  f|  b|  e|         5|
|  f|  g|  e|         1|
+---+---+---+----------+



df3: DataFrame = [A: string, B: string ... 1 more field]
leftSide: DataFrame = [A: string, C: string ... 1 more field]
rightSide: DataFrame = [B: string, C: string ... 1 more field]
df3WithProvenance: DataFrame = [A: string, B: string ... 2 more fields]

Without provenance
```
== Analyzed Logical Plan ==
A: string, B: string, C: string
Sort [A#379 ASC NULLS FIRST, B#770 ASC NULLS FIRST, C#381 ASC NULLS FIRST], true
+- Project [A#379, B#770, C#381]
   +- Project [C#381, A#379, B#770]
      +- Join Inner, (C#381 = C#771)
         :- Project [A#379, C#381]
         :  +- Project [_1#376 AS A#379, _2#377 AS B#380, _3#378 AS C#381]
         :     +- LocalRelation [_1#376, _2#377, _3#378]
         +- Project [B#770, C#771]
            +- Project [_1#766 AS A#769, _2#767 AS B#770, _3#768 AS C#771]
               +- LocalRelation [_1#766, _2#767, _3#768]
```

With provenance
```
== Analyzed Logical Plan ==
A: string, B: string, C: string, provenance: bigint
Sort [A#3 ASC NULLS FIRST, B#1104 ASC NULLS FIRST, C#5 ASC NULLS FIRST], true
+- Project [A#3, B#1104, C#5, provenance#1108L]
   +- Project [C#5, A#3, count_A#1088L, B#1104, count_B#1094L, (count_A#1088L * count_B#1094L) AS provenance#1108L]
      +- Project [C#5, A#3, count_A#1088L, B#1104, count_B#1094L]
         +- Join Inner, (C#5 = C#1105)
            :- Aggregate [A#3, C#5], [A#3, C#5, sum(provenance#675L) AS count_A#1088L]
            :  +- Sort [A#3 ASC NULLS FIRST, B#4 ASC NULLS FIRST, C#5 ASC NULLS FIRST], true
            :     +- Project [A#3, B#4, C#5, count#670L AS provenance#675L]
            :        +- Aggregate [A#3, B#4, C#5], [A#3, B#4, C#5, count(1) AS count#670L]
            :           +- Project [_1#0 AS A#3, _2#1 AS B#4, _3#2 AS C#5]
            :              +- LocalRelation [_1#0, _2#1, _3#2]
            +- Aggregate [B#1104, C#1105], [B#1104, C#1105, sum(provenance#1107L) AS count_B#1094L]
               +- Sort [A#1103 ASC NULLS FIRST, B#1104 ASC NULLS FIRST, C#1105 ASC NULLS FIRST], true
                  +- Project [A#1103, B#1104, C#1105, count#1106L AS provenance#1107L]
                     +- Aggregate [A#1103, B#1104, C#1105], [A#1103, B#1104, C#1105, count(1) AS count#1106L]
                        +- Project [_1#1100 AS A#1103, _2#1101 AS B#1104, _3#1102 AS C#1105]
                           +- LocalRelation [_1#1100, _2#1101, _3#1102]
```


In [39]:
//df.join(df2).withColumn("provenance", df("provenance") * df2("provenance")).drop(df("provenance"), df2("provenance")).show()

### Example $ \pi_{AB}R \bowtie \pi_{BC}R \bigcup \pi_{AC}R \bowtie \pi_{BC}R$  

We can compute the provenance annotation for df4 by summing the provenance annotations from df2 and df3 for each tuple in df4.

We need to use union to avoid double counting the provenance annotations for the tuples that are in both df2 and df3.

We also need to group by the columns and sum the provenance annotations for each tuple in df4.

In [40]:
// Expected results:
// +---+---+---+----------+
// |  A|  B|  C|provenance|
// +---+---+---+----------+
// |  a|  b|  c|         8|
// |  a|  b|  e|        10|
// |  d|  b|  c|        10|
// |  d|  b|  e|        50|
// |  d|  g|  e|         5|
// |  f|  b|  e|         5|
// |  f|  g|  e|         2|
// +---+---+---+----------+

val df4 = df2.union(df3).distinct().orderBy("A", "B", "C")
df4.show()

val df4WithProvenance: DataFrame = df3WithProvenance.union(df2WithProvenance)
                                                    .groupBy("A", "B", "C")
                                                    .agg(sum("provenance").as("provenance"))
                                                    .orderBy("A", "B", "C")

df4WithProvenance.show()


+---+---+---+
|  A|  B|  C|
+---+---+---+
|  a|  b|  c|
|  a|  b|  e|
|  d|  b|  c|
|  d|  b|  e|
|  d|  g|  e|
|  f|  b|  e|
|  f|  g|  e|
+---+---+---+

+---+---+---+----------+
|  A|  B|  C|provenance|
+---+---+---+----------+
|  a|  b|  c|         8|
|  a|  b|  e|        10|
|  d|  b|  c|        10|
|  d|  b|  e|        50|
|  d|  g|  e|         5|
|  f|  b|  e|         5|
|  f|  g|  e|         2|
+---+---+---+----------+



df4: Dataset[Row] = [A: string, B: string ... 1 more field]
df4WithProvenance: DataFrame = [A: string, B: string ... 2 more fields]

### Example $ \pi_{AC} (\pi_{AB}R \bowtie \pi_{BC}R \bigcup \pi_{AC}R \bowtie \pi_{BC}R)$  

In [41]:
// Expected results:
// +---+---+----------+
// |  A|  C|provenance|
// +---+---+----------+
// |  a|  c|         8|
// |  a|  e|        10|
// |  d|  c|        10|
// |  d|  e|        55|
// |  f|  e|         7|
// +---+---+----------+

val df5 = df4.select("A", "C").distinct().orderBy("A", "C")
df5.show()

val df5WithProvenance: DataFrame = df4WithProvenance
    .groupBy("A", "C")
    .agg(sum("provenance").as("provenance"))
    .orderBy("A", "C")
df5WithProvenance.show()


+---+---+
|  A|  C|
+---+---+
|  a|  c|
|  a|  e|
|  d|  c|
|  d|  e|
|  f|  e|
+---+---+

+---+---+----------+
|  A|  C|provenance|
+---+---+----------+
|  a|  c|         8|
|  a|  e|        10|
|  d|  c|        10|
|  d|  e|        55|
|  f|  e|         7|
+---+---+----------+



df5: Dataset[Row] = [A: string, C: string]
df5WithProvenance: DataFrame = [A: string, C: string ... 1 more field]

## Boolean semiring

In [42]:
val dfBool: DataFrame = spark.createDataFrame(
    Seq(
        ("a", "b", "c", false),
        ("d", "b", "e", true),
        ("f", "g", "e", false)
    )
).toDF("A", "B", "C", "provenance")

df.show()

+---+---+---+
|  A|  B|  C|
+---+---+---+
|  a|  b|  c|
|  a|  b|  c|
|  d|  b|  e|
|  d|  b|  e|
|  d|  b|  e|
|  d|  b|  e|
|  d|  b|  e|
|  f|  g|  e|
+---+---+---+



dfBool: DataFrame = [A: string, B: string ... 2 more fields]

In [43]:
val df2: DataFrame = dfBool
    .select("A", "B")
    .join(dfBool.select("B", "C"), "B")
    .select("A", "B", "C")
    .orderBy("A", "B", "C")

df2.show()



val df2WithProvenance: DataFrame = dfBool
    .select(col("A"), col("B"), col("provenance").as("provA"))
    .join(dfBool.select(col("B"), col("C"), col("provenance").as("provC")), Seq("B"))
    .withColumn("provenance", col("provA") && col("provC"))
    .drop("provA", "provC")
    .select("A", "B", "C", "provenance")
    .orderBy("A", "B", "C")

df2WithProvenance.show()


+---+---+---+
|  A|  B|  C|
+---+---+---+
|  a|  b|  c|
|  a|  b|  e|
|  d|  b|  c|
|  d|  b|  e|
|  f|  g|  e|
+---+---+---+

+---+---+---+----------+
|  A|  B|  C|provenance|
+---+---+---+----------+
|  a|  b|  c|     false|
|  a|  b|  e|     false|
|  d|  b|  c|     false|
|  d|  b|  e|      true|
|  f|  g|  e|     false|
+---+---+---+----------+



df2: DataFrame = [A: string, B: string ... 1 more field]
df2WithProvenance: DataFrame = [A: string, B: string ... 2 more fields]

In [44]:
val df3: DataFrame = dfBool
    .select("A", "C")
    .join(dfBool.select("B", "C"), "C")
    .select("A", "B", "C")
    .orderBy("A", "B", "C")

df3.show()

val dfR = dfBool.select(col("A"), col("C"), col("provenance"))
val dfL = dfBool.select(col("B"), col("C"), col("provenance"))

val df3WithProvenance: DataFrame = dfR.withColumnRenamed("provenance", "provA")
    .join(dfL.withColumnRenamed("provenance", "provB"), Seq("C"))
    .withColumn("provenance", col("provA") && col("provB"))
    .drop("provA", "provB")
    .select("A", "B", "C", "provenance")
    .orderBy("A", "B", "C")

df3WithProvenance.show()


+---+---+---+
|  A|  B|  C|
+---+---+---+
|  a|  b|  c|
|  d|  b|  e|
|  d|  g|  e|
|  f|  b|  e|
|  f|  g|  e|
+---+---+---+

+---+---+---+----------+
|  A|  B|  C|provenance|
+---+---+---+----------+
|  a|  b|  c|     false|
|  d|  b|  e|      true|
|  d|  g|  e|     false|
|  f|  b|  e|     false|
|  f|  g|  e|     false|
+---+---+---+----------+



df3: DataFrame = [A: string, B: string ... 1 more field]
dfR: DataFrame = [A: string, C: string ... 1 more field]
dfL: DataFrame = [B: string, C: string ... 1 more field]
df3WithProvenance: DataFrame = [A: string, B: string ... 2 more fields]

In [45]:
val df4 = df2.union(df3).distinct().orderBy("A", "B", "C")

df4.show()

val leftSide = df2WithProvenance.select(col("A"), col("B"), col("C"), lit(true).as("in_df2"))
val rightSide = df3WithProvenance.select(col("A"), col("B"), col("C"), lit(true).as("in_df3"))

val df4WithProvenance = df2WithProvenance
  .union(df3WithProvenance)
  .groupBy("A", "B", "C")
  .agg(max("provenance").as("provenance"))
  .orderBy("A", "B", "C")

df4WithProvenance.show()



+---+---+---+
|  A|  B|  C|
+---+---+---+
|  a|  b|  c|
|  a|  b|  e|
|  d|  b|  c|
|  d|  b|  e|
|  d|  g|  e|
|  f|  b|  e|
|  f|  g|  e|
+---+---+---+

+---+---+---+----------+
|  A|  B|  C|provenance|
+---+---+---+----------+
|  a|  b|  c|     false|
|  a|  b|  e|     false|
|  d|  b|  c|     false|
|  d|  b|  e|      true|
|  d|  g|  e|     false|
|  f|  b|  e|     false|
|  f|  g|  e|     false|
+---+---+---+----------+



df4: Dataset[Row] = [A: string, B: string ... 1 more field]
leftSide: DataFrame = [A: string, B: string ... 2 more fields]
rightSide: DataFrame = [A: string, B: string ... 2 more fields]
df4WithProvenance: Dataset[Row] = [A: string, B: string ... 2 more fields]

In [46]:
val df5 = df4.select("A", "C").distinct().orderBy("A", "C")

df5.show()

// For provenance, we can take the maximum (logical OR) of the provenance values for each (A, C) pair
val df5WithProvenance = df4WithProvenance
    .groupBy("A", "C")
    .agg(max("provenance").as("provenance"))
    .orderBy("A", "C")

df5WithProvenance.show()

+---+---+
|  A|  C|
+---+---+
|  a|  c|
|  a|  e|
|  d|  c|
|  d|  e|
|  f|  e|
+---+---+

+---+---+----------+
|  A|  C|provenance|
+---+---+----------+
|  a|  c|     false|
|  a|  e|     false|
|  d|  c|     false|
|  d|  e|      true|
|  f|  e|     false|
+---+---+----------+



df5: Dataset[Row] = [A: string, C: string]
df5WithProvenance: Dataset[Row] = [A: string, C: string ... 1 more field]

## Why provenance semiring

In [47]:
import java.util.UUID.randomUUID

val dfWhichProv: DataFrame = spark.createDataFrame(
    Seq(
        ("a", "b", "c", Seq(randomUUID().toString)),
        ("d", "b", "e", Seq(randomUUID().toString)),
        ("f", "g", "e", Seq(randomUUID().toString))
    )
).toDF("A", "B", "C", "provenance")

// Set show to false to display full provenance values without truncation
dfWhichProv.show(false)

+---+---+---+--------------------------------------+
|A  |B  |C  |provenance                            |
+---+---+---+--------------------------------------+
|a  |b  |c  |[68223fa6-9ac8-4335-8bab-e346f1574672]|
|d  |b  |e  |[2f1e76e0-92b4-4778-9de1-2e8d44a8f14f]|
|f  |g  |e  |[f2767d6e-bee7-47a5-80c4-c485f8071bf9]|
+---+---+---+--------------------------------------+



import java.util.UUID.randomUUID
dfWhichProv: DataFrame = [A: string, B: string ... 2 more fields]

In [48]:
val dfL = dfWhichProv.select("B", "C", "provenance")
val dfR = dfWhichProv.select("A", "B", "provenance")

// For the join on B, we want to combine the provenance from both sides using array_union to get the unique set of provenance values
val df2WithProvenance: DataFrame = dfL.withColumnRenamed("provenance", "provC")
    .join(dfR.withColumnRenamed("provenance", "provA"), Seq("B"))
    .withColumn("provenance", array_union(col("provA"), col("provC")))
    .drop("provA", "provC")
    .select("A", "B", "C", "provenance")
    .orderBy("A", "B", "C")

df2WithProvenance.show(false)


+---+---+---+----------------------------------------------------------------------------+
|A  |B  |C  |provenance                                                                  |
+---+---+---+----------------------------------------------------------------------------+
|a  |b  |c  |[68223fa6-9ac8-4335-8bab-e346f1574672]                                      |
|a  |b  |e  |[68223fa6-9ac8-4335-8bab-e346f1574672, 2f1e76e0-92b4-4778-9de1-2e8d44a8f14f]|
|d  |b  |c  |[2f1e76e0-92b4-4778-9de1-2e8d44a8f14f, 68223fa6-9ac8-4335-8bab-e346f1574672]|
|d  |b  |e  |[2f1e76e0-92b4-4778-9de1-2e8d44a8f14f]                                      |
|f  |g  |e  |[f2767d6e-bee7-47a5-80c4-c485f8071bf9]                                      |
+---+---+---+----------------------------------------------------------------------------+



dfL: DataFrame = [B: string, C: string ... 1 more field]
dfR: DataFrame = [A: string, B: string ... 1 more field]
df2WithProvenance: DataFrame = [A: string, B: string ... 2 more fields]

In [49]:
val dfL = dfWhichProv.select("B", "C", "provenance")
val dfR = dfWhichProv.select("A", "C", "provenance")

// For the join on C, we want to combine the provenance from both sides using array_union to get the unique set of provenance values
val df3WithProvenance: DataFrame = dfL.withColumnRenamed("provenance", "provB")
    .join(dfR.withColumnRenamed("provenance", "provA"), Seq("C"))
    .withColumn("provenance", array_union(col("provA"), col("provB")))
    .drop("provA", "provB")
    .select("A", "B", "C", "provenance") // Select the columns in the desired order
    .orderBy("A", "B", "C")

df3WithProvenance.show(false)

+---+---+---+----------------------------------------------------------------------------+
|A  |B  |C  |provenance                                                                  |
+---+---+---+----------------------------------------------------------------------------+
|a  |b  |c  |[68223fa6-9ac8-4335-8bab-e346f1574672]                                      |
|d  |b  |e  |[2f1e76e0-92b4-4778-9de1-2e8d44a8f14f]                                      |
|d  |g  |e  |[2f1e76e0-92b4-4778-9de1-2e8d44a8f14f, f2767d6e-bee7-47a5-80c4-c485f8071bf9]|
|f  |b  |e  |[f2767d6e-bee7-47a5-80c4-c485f8071bf9, 2f1e76e0-92b4-4778-9de1-2e8d44a8f14f]|
|f  |g  |e  |[f2767d6e-bee7-47a5-80c4-c485f8071bf9]                                      |
+---+---+---+----------------------------------------------------------------------------+



dfL: DataFrame = [B: string, C: string ... 1 more field]
dfR: DataFrame = [A: string, C: string ... 1 more field]
df3WithProvenance: DataFrame = [A: string, B: string ... 2 more fields]

In [50]:
// For the union of df2 and df3, we want to combine the provenance values for any rows that appear in both using collect_set 
// to get the unique set of provenance values and flatten to combine them into a single array
val df4WithProvenance = df2WithProvenance
  .union(df3WithProvenance)
  .groupBy("A", "B", "C")
  .agg(
    flatten(collect_set(col("provenance"))).as("provenance")
  )
  .orderBy("A", "B", "C")

df4WithProvenance.show(false)



+---+---+---+----------------------------------------------------------------------------+
|A  |B  |C  |provenance                                                                  |
+---+---+---+----------------------------------------------------------------------------+
|a  |b  |c  |[68223fa6-9ac8-4335-8bab-e346f1574672]                                      |
|a  |b  |e  |[68223fa6-9ac8-4335-8bab-e346f1574672, 2f1e76e0-92b4-4778-9de1-2e8d44a8f14f]|
|d  |b  |c  |[2f1e76e0-92b4-4778-9de1-2e8d44a8f14f, 68223fa6-9ac8-4335-8bab-e346f1574672]|
|d  |b  |e  |[2f1e76e0-92b4-4778-9de1-2e8d44a8f14f]                                      |
|d  |g  |e  |[2f1e76e0-92b4-4778-9de1-2e8d44a8f14f, f2767d6e-bee7-47a5-80c4-c485f8071bf9]|
|f  |b  |e  |[f2767d6e-bee7-47a5-80c4-c485f8071bf9, 2f1e76e0-92b4-4778-9de1-2e8d44a8f14f]|
|f  |g  |e  |[f2767d6e-bee7-47a5-80c4-c485f8071bf9]                                      |
+---+---+---+----------------------------------------------------------------------------+

df4WithProvenance: Dataset[Row] = [A: string, B: string ... 2 more fields]

In [51]:
// For the final projection to (A, C), we want to combine the provenance values for any rows that appear in both using collect_set 
// to get the unique set of provenance values and flatten to combine them into a single array 
// We also use array_distinct to remove any duplicate provenance values that may arise from the union
val df5WithProvenance = df4WithProvenance
    .groupBy("A", "C")
    .agg(
        // array_distinct(flatten(collect_list(col("provenance")))).as("provenance")
        min_by(col("provenance"), array_size(col("provenance")))
    )
    .orderBy("A", "C")

df5WithProvenance.show(false)

+---+---+----------------------------------------------------------------------------+
|A  |C  |min_by(provenance, array_size(provenance))                                  |
+---+---+----------------------------------------------------------------------------+
|a  |c  |[68223fa6-9ac8-4335-8bab-e346f1574672]                                      |
|a  |e  |[68223fa6-9ac8-4335-8bab-e346f1574672, 2f1e76e0-92b4-4778-9de1-2e8d44a8f14f]|
|d  |c  |[2f1e76e0-92b4-4778-9de1-2e8d44a8f14f, 68223fa6-9ac8-4335-8bab-e346f1574672]|
|d  |e  |[2f1e76e0-92b4-4778-9de1-2e8d44a8f14f]                                      |
|f  |e  |[f2767d6e-bee7-47a5-80c4-c485f8071bf9]                                      |
+---+---+----------------------------------------------------------------------------+



df5WithProvenance: Dataset[Row] = [A: string, C: string ... 1 more field]

## String provenance

We keep using the type uuid provenance for simplicity, but the same approach would work for any string provenance values as well

We can reuse the same approach for string provenance, just using string concatenation instead of array_union 

In [52]:
val dfL = dfWhichProv.select("B", "C", "provenance")
val dfR = dfWhichProv.select("A", "B", "provenance")

// For the join on B, we want to combine the provenance from both sides using string concatenation with " * " and wrap each provenance in parentheses
// We need to convert the provenance columns (which are arrays) to strings before concatenation
val df2WithStringProvenance: DataFrame = dfL.withColumnRenamed("provenance", "provC")
    .join(dfR.withColumnRenamed("provenance", "provA"), Seq("B"))
    .withColumn(
        "provenance",
        concat(
            lit("("),
            array_join(col("provA"), ","),
            lit(" * "),
            array_join(col("provC"), ","),
            lit(")")
        )
    )
    .drop("provA", "provC")
    .select("A", "B", "C", "provenance")
    .orderBy("A", "B", "C")

df2WithStringProvenance.show(false)




+---+---+---+-----------------------------------------------------------------------------+
|A  |B  |C  |provenance                                                                   |
+---+---+---+-----------------------------------------------------------------------------+
|a  |b  |c  |(68223fa6-9ac8-4335-8bab-e346f1574672 * 68223fa6-9ac8-4335-8bab-e346f1574672)|
|a  |b  |e  |(68223fa6-9ac8-4335-8bab-e346f1574672 * 2f1e76e0-92b4-4778-9de1-2e8d44a8f14f)|
|d  |b  |c  |(2f1e76e0-92b4-4778-9de1-2e8d44a8f14f * 68223fa6-9ac8-4335-8bab-e346f1574672)|
|d  |b  |e  |(2f1e76e0-92b4-4778-9de1-2e8d44a8f14f * 2f1e76e0-92b4-4778-9de1-2e8d44a8f14f)|
|f  |g  |e  |(f2767d6e-bee7-47a5-80c4-c485f8071bf9 * f2767d6e-bee7-47a5-80c4-c485f8071bf9)|
+---+---+---+-----------------------------------------------------------------------------+



dfL: DataFrame = [B: string, C: string ... 1 more field]
dfR: DataFrame = [A: string, B: string ... 1 more field]
df2WithStringProvenance: DataFrame = [A: string, B: string ... 2 more fields]

In [53]:

val dfL = dfWhichProv.select("A", "C", "provenance")
val dfR = dfWhichProv.select("B", "C", "provenance")

// For the join on C, we want to combine the provenance from both sides using concat to create
// a string representation of the provenance with " * " between them and wrap the whole thing in parentheses
val df3WithStringProvenance: DataFrame = dfL.withColumnRenamed("provenance", "provC")
    .join(dfR.withColumnRenamed("provenance", "provB"), Seq("C"))
    .withColumn(
        "provenance",
        concat(
            lit("("),
            array_join(col("provB"), ","),
            lit(" * "),
            array_join(col("provC"), ","),
            lit(")")
        )
    )
    .drop("provB", "provC")
    .select("A", "B", "C", "provenance")
    .orderBy("A", "B", "C")

df3WithStringProvenance.show(false)

+---+---+---+-----------------------------------------------------------------------------+
|A  |B  |C  |provenance                                                                   |
+---+---+---+-----------------------------------------------------------------------------+
|a  |b  |c  |(68223fa6-9ac8-4335-8bab-e346f1574672 * 68223fa6-9ac8-4335-8bab-e346f1574672)|
|d  |b  |e  |(2f1e76e0-92b4-4778-9de1-2e8d44a8f14f * 2f1e76e0-92b4-4778-9de1-2e8d44a8f14f)|
|d  |g  |e  |(f2767d6e-bee7-47a5-80c4-c485f8071bf9 * 2f1e76e0-92b4-4778-9de1-2e8d44a8f14f)|
|f  |b  |e  |(2f1e76e0-92b4-4778-9de1-2e8d44a8f14f * f2767d6e-bee7-47a5-80c4-c485f8071bf9)|
|f  |g  |e  |(f2767d6e-bee7-47a5-80c4-c485f8071bf9 * f2767d6e-bee7-47a5-80c4-c485f8071bf9)|
+---+---+---+-----------------------------------------------------------------------------+



dfL: DataFrame = [A: string, C: string ... 1 more field]
dfR: DataFrame = [B: string, C: string ... 1 more field]
df3WithStringProvenance: DataFrame = [A: string, B: string ... 2 more fields]

In [54]:
// For the union of df2 and df3, we want to combine the provenance values for any rows that appear in both using collect_list
// to display all provenance values in a string with " + " between them and wrap the whole thing in parentheses

val df4WithStringProvenance = df2WithStringProvenance
  .union(df3WithStringProvenance)
  .groupBy("A", "B", "C")
  .agg(
      concat(
          lit("("),
          array_join(collect_list(col("provenance")), " + "),
          lit(")")
      ).as("provenance")
  )
  .orderBy("A", "B", "C")

df4WithStringProvenance.show(false)



+---+---+---+---------------------------------------------------------------------------------------------------------------------------------------------------------------+
|A  |B  |C  |provenance                                                                                                                                                     |
+---+---+---+---------------------------------------------------------------------------------------------------------------------------------------------------------------+
|a  |b  |c  |((68223fa6-9ac8-4335-8bab-e346f1574672 * 68223fa6-9ac8-4335-8bab-e346f1574672) + (68223fa6-9ac8-4335-8bab-e346f1574672 * 68223fa6-9ac8-4335-8bab-e346f1574672))|
|a  |b  |e  |((68223fa6-9ac8-4335-8bab-e346f1574672 * 2f1e76e0-92b4-4778-9de1-2e8d44a8f14f))                                                                                |
|d  |b  |c  |((2f1e76e0-92b4-4778-9de1-2e8d44a8f14f * 68223fa6-9ac8-4335-8bab-e346f1574672))                                      

df4WithStringProvenance: Dataset[Row] = [A: string, B: string ... 2 more fields]

In [55]:
// For the final projection to (A, C), we want to combine the provenance values for any rows that appear in both using collect_list
// to display all provenance values in a string with " + " between them and wrap the whole thing in parentheses
val df5WithStringProvenance = df4WithStringProvenance       
    .groupBy("A", "C")
    .agg(
      concat(
          lit("("),
          array_join(collect_list(col("provenance")), " + "),
          lit(")")
      ).as("provenance")
    )
    .orderBy("A", "C")

df5WithStringProvenance.show(false)

+---+---+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|A  |C  |provenance                                                                                                                                                                                                                                         |
+---+---+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|a  |c  |(((68223fa6-9ac8-4335-8bab-e346f1574672 * 68223fa6-9ac8-4335-8bab-e346f1574672) + (68223fa6-9ac8-4335-8bab-e346f1574672 * 68223fa6-9ac8-4335-8bab-e346f1574672)))                                                                    

df5WithStringProvenance: Dataset[Row] = [A: string, C: string ... 1 more field]